In [14]:
import os
from typing import TypedDict, List, Dict, Any, Optional, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, RemoveMessage
from typing_extensions import Literal
from langgraph.graph import MessagesState
from pydantic import BaseModel, Field
import sqlite3

In [5]:
from typing import TypedDict


class State(TypedDict, total=False):
    user_input: str
    first_name: str
    middle_name: str
    last_name: str
    confirmation: str
    email: str

In [7]:
def get_name(state: State):

    user_input = input(
        "Please provide your name in this order and format: "
        "Firstname Middlename Lastname: "
    )

    first_name, middle_name, last_name = user_input.split()

    return {
        "user_input": user_input,
        "first_name": first_name,
        "middle_name": middle_name,
        "last_name": last_name
    }

In [9]:
def confirmer(state: State):

    print(
        "\nPlease confirm your name:"
        f"\nFirst name: {state['first_name']}"
        f"\nMiddle name: {state['middle_name']}"
        f"\nLast name: {state['last_name']}"
    )

    confirmation = input(
        "\nIs this correct? (Yes/No): "
    )

    return {
        "confirmation": confirmation
    }

In [10]:
def confirmation_router(state: State):

    confirmation = state["confirmation"].strip().lower()

    if confirmation in ["yes", "y", "yeah", "yep", "correct", "right"]:
        return "get_email"

    elif confirmation in ["no", "n", "nope", "wrong", "incorrect"]:
        return "get_name"

    else:
        return "confirmer"

In [11]:
def get_email(state: State):

    print("\n>>> SUCCESS: Moving to get_email node.")

    return {}

In [ ]:
def execute_query(query, params=()):
    with sqlite3.connect("Onboardingg.db") as conn:
        cursor = conn.cursor()
        cursor.execute(query, params)
        conn.commit()



def insert_profile_to_db(State):
    query = f"INSERT INTO profile (first_name, middle_name, last_name) VALUES (?, ?, ?)"
    params = (state.get("first_name"), state.get("middle_name"), state.get("last_name"))
    execute_query(query, params)
    return state

In [12]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(State)


# Nodes
builder.add_node("get_name", get_name)
builder.add_node("confirmer", confirmer)
builder.add_node("get_email", get_email)
builder.add_node("insert_profile_to_db", insert_profile_to_db)


# Start
builder.add_edge(START, "get_name")


# get_name → confirmer
builder.add_edge("get_name", "confirmer")


# confirmer → conditional routing
builder.add_conditional_edges(
    "confirmer",
    confirmation_router,
    {
        "get_email": "get_email",
        "get_name": "get_name",
        "confirmer": "confirmer"
    }
)


# Temporary end
builder.add_edge("get_email", "insert_profile_to_db")
builder.add_edge("insert_profile_to_db", END)


graph = builder.compile()

In [15]:
graph.invoke({})


Please confirm your name:
First name: A
Middle name: M
Last name: l

>>> SUCCESS: Moving to get_email node.


{'user_input': ' A M l',
 'first_name': 'A',
 'middle_name': 'M',
 'last_name': 'l',
 'confirmation': 'yes'}